In [1]:
# groq api key
# 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
import os
os.environ["GROQ_API_KEY"] = "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"

import os
import pandas as pd
import numpy as np

import nest_asyncio
nest_asyncio.apply()


In [ ]:
!pip install llama-index
!pip install llama-index-llms-groq
!pip install llama-index-embeddings-huggingface
!pip install llama-parse
!pip install nest_asyncio

In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Initialize the embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")


/home/aya/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Load documents from the 'data' directory
documents = SimpleDirectoryReader('./data').load_data()

# Better option

from llama_index.core.node_parser import SimpleFileNodeParser
from llama_index.readers.file import FlatReader
from pathlib import Path
# md_docs = FlatReader().load_data(Path("./data"))

parser = SimpleFileNodeParser()
md_nodes = parser.get_nodes_from_documents(documents=documents)
md_nodes

Ignoring wrong pointing object 84 0 (offset 0)


[TextNode(id_='e6db320c-7b0f-4b2a-adce-56e16f4cf90f', embedding=None, metadata={'page_label': '1', 'file_name': 'Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_path': '/home/aya/genai_expts/data/Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_type': 'application/pdf', 'file_size': 71542, 'creation_date': '2025-04-26', 'last_modified_date': '2025-04-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='fce05c53-3bc8-4e4b-bec0-83c165243c25', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_path': '/home/aya/genai_expts/data/Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_type': 'application/pdf

In [4]:
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser

sent_splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=20,
)
sem_splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)


sent_nodes = sent_splitter.get_nodes_from_documents(documents)
sem_nodes = sem_splitter.abuild_semantic_nodes_from_documents(documents, show_progress= False)


In [26]:
documents

[Document(id_='fce05c53-3bc8-4e4b-bec0-83c165243c25', embedding=None, metadata={'page_label': '1', 'file_name': 'Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_path': '/home/aya/genai_expts/data/Letter-from-FB-Engineer-regarding-interviews.pdf', 'file_type': 'application/pdf', 'file_size': 71542, 'creation_date': '2025-04-26', 'last_modified_date': '2025-04-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text=" \n1601 Willow Road, Menlo Park, CA  94025  \n    Dear fellow engineer,  I'm an engineer here at Facebook and I want to share with you some thoughts on our interview process.  Our interview process is designed around the id

In [ ]:
# Simple VectorStoreIndex
from llama_index.core import VectorStoreIndex

# Create the vector index
index = VectorStoreIndex(sem_nodes, embed_model=embed_model)

# Persist the index for future use
index.storage_context.persist(persist_dir="./storage")

In [3]:
from llama_index.llms.groq import Groq

# Initialize the Groq LLM
llm = Groq(model="llama3-8b-8192")


In [4]:
import chromadb
# save to disk
# db = chromadb.PersistentClient(path="./chroma_db")
# load from disk
db = chromadb.PersistentClient(path="./chroma_db")

In [5]:
#ChromaDB
# !pip install llama-index chromadb --quiet
# !pip install chromadb
# !pip install sentence-transformers
# !pip install pydantic

from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from IPython.display import Markdown, display
from llama_index.core.query_engine import RetrieverQueryEngine

# Load documents from the 'data' directory
documents = SimpleDirectoryReader('./data').load_data()


# create client and a new collection
# chroma_client = chromadb.EphemeralClient()
# Create this one time
chroma_collection = db.get_or_create_collection("test_db")


# set up ChromaVectorStore and load in data
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, embed_model=embed_model
)

Ignoring wrong pointing object 84 0 (offset 0)


In [6]:

# Initialize the query engine
retriever = index.as_retriever(similarity_top_k=3)
query_engine = RetrieverQueryEngine.from_args(retriever, llm=llm)


In [7]:

response = query_engine.query("Who is Ayanava Sarkar?")
display(Markdown(f"<b>{response}</b>"))

<b>ML Engineer (5+ years) specializing in GenAI, LLMs, and Multimodal AI with 100+ research citations.</b>

In [8]:
doc_to_update = chroma_collection.get()
doc_to_update

{'ids': ['47ed8c58-159a-4c62-a3d5-26c1bdac5934',
  '9316569a-7f69-49de-80cf-6fe1ac35b822',
  '4505efdf-c56c-476b-9664-86e8f77efd54'],
 'embeddings': None,
 'documents': ["1601 Willow Road, Menlo Park, CA  94025  \n    Dear fellow engineer,  I'm an engineer here at Facebook and I want to share with you some thoughts on our interview process.  Our interview process is designed around the idea that all programmers that we want to hire, with proper preparation, have the ability to quickly whiteboard solutions to questions that involve describing an algorithm and translating that description into working code.  Doing well on this is a strong signal that you're able to understand how to write efficient algorithms, effectively problem-solve, and communicate your thoughts in code clearly.  Unfortunately, we find that many good engineers come to our interviews without preparing for these types of questions.  While we think that any good engineer can with practice perform well on these types of 

In [28]:
print(doc_to_update.keys())
print(doc_to_update['documents'][0])

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
1601 Willow Road, Menlo Park, CA  94025  
    Dear fellow engineer,  I'm an engineer here at Facebook and I want to share with you some thoughts on our interview process.  Our interview process is designed around the idea that all programmers that we want to hire, with proper preparation, have the ability to quickly whiteboard solutions to questions that involve describing an algorithm and translating that description into working code.  Doing well on this is a strong signal that you're able to understand how to write efficient algorithms, effectively problem-solve, and communicate your thoughts in code clearly.  Unfortunately, we find that many good engineers come to our interviews without preparing for these types of questions.  While we think that any good engineer can with practice perform well on these types of coding questions, it's difficult to get a strong signal for someone who did not prepa

In [9]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

nodes = retriever.retrieve("who is Ayanava?")
# nodes
for node in nodes:
    print(f"ID: {node.node.ref_doc_id} | Score: {node.score:.2f}")
    print(f"Text: {node.node.text[:100]}...\n")

ID: be921a1f-190e-45b9-b137-60f9d1a2d465 | Score: 0.18
Text: AYANAVA  SARKAR  ayanavasarka@gmail.com  ||  (408)791-7150  ||   Bay  Area,  CA  Github  ||  LinkedI...

ID: be921a1f-190e-45b9-b137-60f9d1a2d465 | Score: 0.17
Text: ➢  Optimized  polyp  detection  by  over  52%  for  Iterative’s  flagship  product  by  using  quant...

ID: cf589472-906d-461e-8e88-93a7b9c765b9 | Score: 0.13
Text: 1601 Willow Road, Menlo Park, CA  94025  
    Dear fellow engineer,  I'm an engineer here at Faceboo...



In [53]:
# Get the most relevant node
target_node = nodes[0]  # Top result
chroma_id = target_node.node.ref_doc_id
old_text = target_node.node.text
print(chroma_id)

1e10b079-0cb6-4ddc-ba14-00194c6502de


In [38]:
response = query_engine.query("What is the address of Ayanava?")
response.response

'1601 Willow Road, Menlo Park, CA 94025'

In [37]:
response.source_nodes[0].node_id

AttributeError: 'AgentOutput' object has no attribute 'source_nodes'

In [36]:
# chroma_collection.upsert(
#     ids=[response.source_nodes[0].node_id],
#     documents=["San Jose"],
#     metadatas=[{"source": "updated_source"}]
# )


# Step 1: Define the LLM-based Replacement Function
def replace_phrase_with_llm(prompt) -> str:
    """
    Send a prompt to the LLM asking it to replace all occurrences of phrase1 with phrase2.
    """
    
    # Call the LLM
    response = llm.complete(prompt)
    return response.text.strip()

from llama_index.core.agent import AgentRunner, Task
from llama_index.core.agent.workflow import FunctionAgent

# from llama_index.core.tools.function_tool import FunctionTool
from llama_index.core.agent.workflow import AgentWorkflow

import nest_asyncio
nest_asyncio.apply()

# Example Inputs
large_text = "The quick brown fox jumps over the lazy dog. The fox is very quick."
phrase1 = "fox"
phrase2 = "cat"

prompt = f"""
    You are a helpful assistant.
        Here is a large text:

    {large_text}
    
    Please replace **every occurrence** of the phrase:
    - "{phrase1}"

    with the phrase:
    - "{phrase2}"

    Make sure:
    - Preserve the original formatting.
    - Do not change anything else.
    - Output the entire updated text only, without any additional commentary.
    """
    
workflow = FunctionAgent(
    tools=[replace_phrase_with_llm],
    llm=llm,
    )

from llama_index.core.llms import ChatMessage, ImageBlock, TextBlock

msg = ChatMessage(
    role="user",
    blocks=[
        TextBlock(text=prompt),
        # ImageBlock(path="./screenshot.png"),
    ],
)

response = await workflow.run(msg)

print("Updated Text:\n", response)

Updated Text:
 The quick brown cat jumps over the lazy dog. The cat is very quick.


In [13]:
result = chroma_collection.get(ids=[response.source_nodes[0].node_id])
print(result["documents"][0])

# Test again with query
results = chroma_collection.query(query_texts=["What is the address of Ayanava?"], n_results=1)
print(results["documents"][0])


San Jose
['San Jose']


In [14]:
chroma_collection.get()

{'ids': ['47ed8c58-159a-4c62-a3d5-26c1bdac5934',
  '9316569a-7f69-49de-80cf-6fe1ac35b822',
  '4505efdf-c56c-476b-9664-86e8f77efd54'],
 'embeddings': None,
 'documents': ["1601 Willow Road, Menlo Park, CA  94025  \n    Dear fellow engineer,  I'm an engineer here at Facebook and I want to share with you some thoughts on our interview process.  Our interview process is designed around the idea that all programmers that we want to hire, with proper preparation, have the ability to quickly whiteboard solutions to questions that involve describing an algorithm and translating that description into working code.  Doing well on this is a strong signal that you're able to understand how to write efficient algorithms, effectively problem-solve, and communicate your thoughts in code clearly.  Unfortunately, we find that many good engineers come to our interviews without preparing for these types of questions.  While we think that any good engineer can with practice perform well on these types of 

In [15]:
query_engine.query("WHere does Ayanava stay?")

Response(response='San Jose', source_nodes=[NodeWithScore(node=TextNode(id_='9316569a-7f69-49de-80cf-6fe1ac35b822', embedding=None, metadata={'page_label': '1', 'file_name': 'Resume_AYANAVA_ent.pdf', 'file_path': '/home/aya/genai_expts/data/Resume_AYANAVA_ent.pdf', 'file_type': 'application/pdf', 'file_size': 222027, 'creation_date': '2025-04-26', 'last_modified_date': '2025-04-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='be921a1f-190e-45b9-b137-60f9d1a2d465', node_type='4', metadata={'page_label': '1', 'file_name': 'Resume_AYANAVA_ent.pdf', 'file_path': '/home/aya/genai_expts/data/Resume_AYANAVA_ent.pdf', 'file_type': 'application/pdf', 'file_size': 222027, 'creation_date': '2025-04-26', 'last_mod